# NSCLC Population Sampling

This notebook samples NSCLC populations using the current stepwise design:

1. Sample the shared entry graph in `nsclc.yaml`.
2. Read `condition_episode.pathway`.
3. Route the sampled person into the matching stage-specific tail graph.
4. Collect the combined state, active treatment endpoints, and normalized regimen outcome.

The main goal is quick clinical face-validity review of routed populations and endpoint distributions.

In [1]:
from pathlib import Path
import sys
import random

import pandas as pd


def find_repo_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "scoop").exists() and (candidate / "notebooks").exists():
            return candidate
    raise RuntimeError("Could not locate the SCOOP repo root.")


ROOT = find_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

CONFIG_ROOT = ROOT / "scoop" / "configuration"

from scoop.configuration.config_loader import load_profile_manifest, load_routed_profile


2026-04-06 22:00:31,555 | DEBUG   | scoop | SCOOP logging initialized


In [2]:
manifest = load_profile_manifest("nsclc", base_dir=CONFIG_ROOT)
runtime = load_routed_profile("nsclc", base_dir=CONFIG_ROOT)

{
    "profile_id": manifest.profile_id,
    "default_graph": manifest.default_graph,
    "entry_graph": runtime.entry_graph_name,
    "tail_graphs": sorted(runtime.tail_contexts),
    "pathway_to_graph": runtime.pathway_to_graph,
}


{'profile_id': 'nsclc',
 'default_graph': 'nsclc',
 'entry_graph': 'nsclc',
 'tail_graphs': ['nsclc_stg1',
  'nsclc_stg23_resectable',
  'nsclc_stg3_unresectable',
  'nsclc_stg4_driver',
  'nsclc_stg4_nodrivers'],
 'pathway_to_graph': {'nsclc_stage1': 'nsclc_stg1',
  'nsclc_stage23_resectable': 'nsclc_stg23_resectable',
  'nsclc_stage3_unresectable': 'nsclc_stg3_unresectable',
  'nsclc_stage4_driver': 'nsclc_stg4_driver',
  'nsclc_stage4_nodrivers': 'nsclc_stg4_nodrivers'}}

In [3]:
def sample_routed_nsclc_population(
    n: int = 1000,
    seed: int | None = 17,
    *,
    include_other: bool = True,
):
    if seed is not None:
        random.seed(seed)

    rows: list[dict] = []
    for sample_index in range(n):
        routed = runtime.sample_routed_patient()

        if routed.tail_graph_name is None and not include_other:
            continue

        graph_outcome = routed.graph_outcome
        active_endpoints = graph_outcome.active_endpoints if graph_outcome is not None else tuple()
        primary_category_id = graph_outcome.primary_category_id if graph_outcome is not None else None
        graph_alias_used = graph_outcome.graph_alias_used if graph_outcome is not None else None

        row = {
            "sample_index": sample_index,
            "entry_graph": routed.entry_graph_name,
            "pathway": routed.pathway,
            "tail_graph_name": routed.tail_graph_name,
            "active_endpoints": active_endpoints,
            "active_endpoint_count": len(active_endpoints),
            "primary_category_id": primary_category_id,
            "graph_alias_used": graph_alias_used,
            "outcome_error": routed.outcome_error,
        }
        row.update(routed.final_sample)
        rows.append(row)

    return pd.DataFrame(rows)


def sample_balanced_pathway_populations(
    n_per_pathway: int = 250,
    seed: int | None = 17,
    *,
    max_draws: int = 100000,
):
    if seed is not None:
        random.seed(seed)

    targets = {pathway: n_per_pathway for pathway in runtime.pathway_to_graph}
    counts = {pathway: 0 for pathway in runtime.pathway_to_graph}
    rows: list[dict] = []
    draws = 0

    while any(counts[p] < targets[p] for p in targets):
        draws += 1
        if draws > max_draws:
            raise RuntimeError(
                f"Reached max_draws={max_draws} before filling all pathways: {counts}"
            )

        routed = runtime.sample_routed_patient()
        if routed.tail_graph_name is None:
            continue
        if counts[routed.pathway] >= targets[routed.pathway]:
            continue

        graph_outcome = routed.graph_outcome
        active_endpoints = graph_outcome.active_endpoints if graph_outcome is not None else tuple()
        primary_category_id = graph_outcome.primary_category_id if graph_outcome is not None else None
        graph_alias_used = graph_outcome.graph_alias_used if graph_outcome is not None else None

        row = {
            "sample_index": counts[routed.pathway],
            "entry_graph": routed.entry_graph_name,
            "pathway": routed.pathway,
            "tail_graph_name": routed.tail_graph_name,
            "active_endpoints": active_endpoints,
            "active_endpoint_count": len(active_endpoints),
            "primary_category_id": primary_category_id,
            "graph_alias_used": graph_alias_used,
            "outcome_error": routed.outcome_error,
        }
        row.update(routed.final_sample)
        rows.append(row)
        counts[routed.pathway] += 1

    df = pd.DataFrame(rows)
    return df, counts, draws


In [4]:
population = sample_routed_nsclc_population(n=3000, seed=17)
population.shape

(3000, 84)

In [5]:
columns = [
    "sample_index",
    "pathway",
    "tail_graph_name",
    "active_endpoints",
    "primary_category_id",
    "outcome_error",
    "condition_occurrence.cancer_stage",
    "condition_episode.group_stage",
    "condition_episode.resectability",
    "condition_occurrence.morphology",
    "condition_occurrence.biomarker",
    "condition_occurrence.pdl1",
    "person.performance_status",
]

population[columns].head(20)


,sample_index,pathway,tail_graph_name,active_endpoints,primary_category_id,outcome_error,condition_occurrence.cancer_stage,condition_episode.group_stage,condition_episode.resectability,condition_occurrence.morphology,condition_occurrence.biomarker,condition_occurrence.pdl1,person.performance_status
0,0,nsclc_stage23_resectable,nsclc_stg23_resectable,"(neoadjuvant_chemo_ici,)",neoadjuvant_chemo_ici,NaN,stage_ii,locally_advanced,operable,nonsquamous,none,absent,ps1
1,1,nsclc_stage1,nsclc_stg1,"(surgery_only,)",surgery_only,NaN,stage_i,early,operable,nonsquamous,none,low,ps2
2,2,nsclc_stage1,nsclc_stg1,"(surgery_only,)",surgery_only,NaN,stage_i,early,operable,nonsquamous,none,low,ps0
3,3,nsclc_stage4_nodrivers,nsclc_stg4_nodrivers,"(ici_w_chemo_squamous,)",ici_w_chemo_squamous,NaN,stage_iv,metastatic,inoperable,squamous,none,absent,ps0
4,4,nsclc_stage23_resectable,nsclc_stg23_resectable,"(neoadjuvant_chemo_ici,)",neoadjuvant_chemo_ici,NaN,stage_ii,locally_advanced,operable,nonsquamous,none,high,ps0
5,5,nsclc_stage3_unresectable,nsclc_stg3_unresectable,"(concurrent_crt_adjuvant_durvalumab,)",concurrent_crt_adjuvant_durvalumab,NaN,stage_ii,locally_advanced,inoperable,nonsquamous,none,high,ps1
6,6,nsclc_stage4_nodrivers,nsclc_stg4_nodrivers,(ici_montherapy_or_ici_with_chemotherapy_nonsq...,ici_monotherapy_or_ici_with_chemotherapy_nonsq...,NaN,stage_iv,metastatic,inoperable,nonsquamous,kras,high,ps0
7,7,nsclc_stage1,nsclc_stg1,"(surgery_only,)",surgery_only,NaN,stage_i,early,operable,nonsquamous,none,low,ps1
8,8,nsclc_stage3_unresectable,nsclc_stg3_unresectable,"(concurrent_crt_adjuvant_durvalumab,)",concurrent_crt_adjuvant_durvalumab,NaN,stage_ii,locally_advanced,inoperable,nonsquamous,none,low,ps0
9,9,nsclc_stage3_unresectable,nsclc_stg3_unresectable,"(concurrent_crt_adjuvant_durvalumab,)",concurrent_crt_adjuvant_durvalumab,NaN,stage_iii,locally_advanced,inoperable,nonsquamous,none,high,ps0


In [6]:
pathway_summary = (
    population.groupby(["pathway", "tail_graph_name"], dropna=False)
    .size()
    .rename("n")
    .reset_index()
    .sort_values(["n", "pathway"], ascending=[False, True])
)

pathway_summary


,pathway,tail_graph_name,n
0,nsclc_stage1,nsclc_stg1,949
1,nsclc_stage23_resectable,nsclc_stg23_resectable,731
2,nsclc_stage3_unresectable,nsclc_stg3_unresectable,697
4,nsclc_stage4_nodrivers,nsclc_stg4_nodrivers,519
3,nsclc_stage4_driver,nsclc_stg4_driver,104


In [7]:
endpoint_summary = (
    population[population["tail_graph_name"].notna()]
    .explode("active_endpoints")
    .groupby(["tail_graph_name", "active_endpoints"], dropna=False)
    .size()
    .rename("n")
    .reset_index()
    .sort_values(["tail_graph_name", "n", "active_endpoints"], ascending=[True, False, True])
)

endpoint_summary


,tail_graph_name,active_endpoints,n
3,nsclc_stg1,surgery_only,573
0,nsclc_stg1,port,245
2,nsclc_stg1,sbrt,94
1,nsclc_stg1,port_adjuvant_osimertinib,33
4,nsclc_stg1,NaN,4
10,nsclc_stg23_resectable,neoadjuvant_chemo_ici,345
5,nsclc_stg23_resectable,adjuvant_atezolizumab_pembrolizumab,189
9,nsclc_stg23_resectable,adjuvant_pembrolizumab,111
6,nsclc_stg23_resectable,adjuvant_cisplatin_osimertinib,69
8,nsclc_stg23_resectable,adjuvant_osimertinib,9


In [8]:
unresolved = population[population["outcome_error"].notna()]
unresolved[[
    c
    for c in [
        "pathway",
        "tail_graph_name",
        "active_endpoints",
        "outcome_error",
    ]
    if c in unresolved.columns
]].drop_duplicates().sort_values(["pathway", "outcome_error"])


,pathway,tail_graph_name,active_endpoints,outcome_error
530,nsclc_stage1,nsclc_stg1,(),No active treatment_episode endpoints found in...


In [9]:
balanced_population, balanced_counts, draws_used = sample_balanced_pathway_populations(
    n_per_pathway=250,
    seed=17,
)

{
    "balanced_counts": balanced_counts,
    "entry_draws_used": draws_used,
    "shape": balanced_population.shape,
}


{'balanced_counts': {'nsclc_stage1': 250,
  'nsclc_stage23_resectable': 250,
  'nsclc_stage3_unresectable': 250,
  'nsclc_stage4_driver': 250,
  'nsclc_stage4_nodrivers': 250},
 'entry_draws_used': 7292,
 'shape': (1250, 84)}

In [10]:
balanced_endpoint_pivot = (
    balanced_population.explode("active_endpoints")
    .pivot_table(
        index="tail_graph_name",
        columns="active_endpoints",
        values="sample_index",
        aggfunc="count",
        fill_value=0,
    )
    .sort_index()
)

balanced_endpoint_pivot


active_endpoints,adjuvant_atezolizumab_pembrolizumab,adjuvant_cisplatin_osimertinib,adjuvant_optchemo_alectinib,adjuvant_osimertinib,adjuvant_pembrolizumab,alk_intervention,best_supportive_care,braf_intervention,chemotherapy_or_atezolizumab_nonsquamous,chemotherapy_or_atezolizumab_squamous,...,port,port_adjuvant_osimertinib,radiotherapy,ret_intervention,ros1_intervention,sbrt,sequential_crt,sequential_crt_adjuvant_durvalumab,sequential_crt_adjuvant_osimertinib,surgery_only
tail_graph_name,,,,,,,,,,,,,,,,,,,,,
nsclc_stg1,0,0,0,0,0,0,0,0,0,0,...,74,4,0,0,0,23,0,0,0,148
nsclc_stg23_resectable,62,19,2,5,35,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
nsclc_stg3_unresectable,0,0,0,0,0,0,0,0,0,0,...,0,0,20,0,0,0,6,17,2,0
nsclc_stg4_driver,0,0,0,0,0,18,0,1,0,0,...,0,0,0,5,2,0,0,0,0,0
nsclc_stg4_nodrivers,0,0,0,0,0,0,18,0,24,3,...,0,0,0,0,0,0,0,0,0,0
